# A Motivating Example: the Bigram Language Model

Open in [Google Colab](https://colab.research.google.com/github/febse/ta2025/blob/main/03-02-ANN-Bigram-Motivation.ipynb)

Until now we have treated text as a *bag of words*: the representations we used (counts, TF-IDF) only looked at *which* words appear in a document, but not at *where* they appear or in *what order*. In this chapter we take the first step towards models that use word order. We start with a very simple probabilistic model of language, the **bigram model**, and use it as a running example that motivates neural networks and, later in the course, word embeddings.

## Why is such a simple model interesting?

The question that the bigram model answers is: *given the current word, how likely is each possible next word?* Even this tiny question is surprisingly powerful, because

- it lets us **score** sentences: a sentence that a speaker of the language could plausibly say gets a higher probability than a word salad (useful, for example, in speech recognition, machine translation and spelling correction);
- it lets us **generate** text by repeatedly sampling the next word;
- it is a **prediction problem with many classes** (one per word in the vocabulary). Training a model to predict the next word is exactly the task on which modern language models are built, so it is the simplest member of a long family of models;
- its weakness (it needs a huge table of probabilities that cannot be estimated reliably and treats all words as unrelated) is precisely what neural networks and word embeddings fix.

The rest of this chapter first builds the model by counting, then looks at its limitations, and finally sketches how a neural network can learn the same table. The code is collected in the last section.

## The probability of a sentence

Let's say that we want to score sentences based on how likely or unlikely they are to appear in a language (for example to detect non-sensical sentences). As an example we use the first sentence of *Alice's Adventures in Wonderland*:

> Alice was beginning to get very tired of sitting by her sister on the bank.

One simple approach could be to look at the frequency of each word in a large corpus of text and then multiply the frequencies of the words in the sentence to get an overall score. However, this is nothing more than the bag-of-words model and it ignores the order of the words. Another approach would be to treat the whole sentence as a single token and look up its frequency in a large corpus. However, you can imagine that most sentences will not appear in the corpus more than once and it will be difficult to estimate their probabilities.

A compromise between these two extremes is to look at sequences of several words, estimate the probabilities of these sequences and then combine them to get the probability of the whole sentence using the chain rule of probability.

Let's look at **Alice was beginning to get very tired** (a shorter example to save space). For events $A$ and $B$, the probability of both happening is

$$
P(A \cap B) = P(A|B) P(B)
$$

and for three events $A$, $B$ and $C$ this extends to

$$
P(A \cap B \cap C) = P(A|B \cap C) P(B|C) P(C)
$$

Applying this to our sentence, we get

$$
\begin{align*}
P(\text{Alice was beginning to get very tired}) & = P(\text{tired}|\text{Alice was beginning to get very})\\
& \times  P(\text{very}|\text{Alice was beginning to get}) \\
& \times P(\text{get}|\text{Alice was beginning to}) \\
& \times P(\text{to}|\text{Alice was beginning}) \\
& \times P(\text{beginning}|\text{Alice was}) \\
& \times P(\text{was}|\text{Alice}) \\
& \times P(\text{Alice})
\end{align*}
$$

Until now we have not made any assumptions about the dependencies between the words, we have simply restated the sentence probability using the chain rule. However, estimating these conditional probabilities is difficult because the number of possible combinations of words is very large and we will not have enough data to estimate them accurately.

## The Markov assumption

We can get around this problem by making a Markov assumption: the conditional probability of a word only depends on the previous word (this is called a first-order Markov assumption). With this assumption the probabilities above simplify to

$$
\begin{align*}
P(\text{Alice was beginning to get very tired}) & = P(\text{tired}|\text{very})\\
& \times  P(\text{very}|\text{get}) \\
& \times P(\text{get}|\text{to}) \\
& \times P(\text{to}|\text{beginning}) \\
& \times P(\text{beginning}|\text{was}) \\
& \times P(\text{was}|\text{Alice}) \\
& \times P(\text{Alice})
\end{align*}
$$

These pairs of consecutive words are called **bigrams**. They are much easier to estimate from data: we count how many times each word follows another word in a large corpus of text and divide by the number of times the first word appears:

$$
\hat{P}(w_t | w_{t-1}) = \frac{C(w_{t-1}, w_t)}{C(w_{t-1})}
$$

where $C(\cdot)$ denotes a count in the corpus. (This is the maximum likelihood estimate of the conditional probabilities.)

## The bigram matrix

We can arrange the bigram probabilities in a matrix, where the rows correspond to the first word and the columns correspond to the second word. Each entry contains the probability of the second word given the first word, so every row sums to one (a row is a probability distribution over the next word). For the tiny "corpus" consisting only of our one sentence the matrix is:

|           | Alice | was | beginning | to  | get | very | tired |
|-----------|-------|-----|-----------|-----|-----|------|-------|
| Alice     | 0     | 1.0 | 0         | 0   | 0   | 0    | 0     |
| was       | 0     | 0   | 1.0       | 0   | 0   | 0    | 0     |
| beginning | 0     | 0   | 0         | 1.0 | 0   | 0    | 0     |
| to        | 0     | 0   | 0         | 0   | 1.0 | 0    | 0     |
| get       | 0     | 0   | 0         | 0   | 0   | 1.0  | 0     |
| very      | 0     | 0   | 0         | 0   | 0   | 0    | 1.0   |
| tired     | 0     | 0   | 0         | 0   | 0   | 0    | 0     |

The last row is empty because "tired" is never followed by another word in this tiny corpus. With a real corpus the table is much larger and its entries are less extreme. The last section of this chapter estimates the matrix from the whole book.

## Using the model: scoring and generating text

With the bigram probabilities at hand we can

- compute the probability of any sentence as $P(w_1) \prod_{t=2}^{T} P(w_t | w_{t-1})$ (in practice we add the logarithms of the probabilities, so that the product of many small numbers does not underflow), and
- generate text by drawing the first word, then repeatedly drawing the next word from the row of the matrix that belongs to the current word.

## Limitations: why counting is not enough

1. **Sparsity.** A vocabulary of $V$ words has $V^2$ possible bigrams (and $V^3$ trigrams, ...). Most of them never appear in a corpus, so their estimated probability is exactly zero, and every sentence that contains an unseen bigram gets probability zero, even if it is perfectly sensible. Smoothing methods (adding pseudo-counts, backing off to shorter contexts) patch this problem, but only partly.
2. **No notion of similarity.** To the counting model, the words "cat" and "dog" are as different as "cat" and "although": each word is a separate row of the table and seeing "the cat sat" tells us nothing about "the dog sat". In the one-hot representation all word vectors are orthogonal.
3. **Short memory.** Conditioning on one previous word ignores almost all of the context. Longer contexts (trigrams, ...) make the sparsity problem exponentially worse.

## From counting to learning

Instead of counting, we can pose next-word prediction as a **classification problem** and learn the table with a neural network. The input is the current word, represented as a one-hot (row) vector $x$ of length $V$, and the output is a vector of $V$ probabilities for the next word. The simplest model computes $V$ logits as a linear function of the input and turns them into probabilities with the softmax function:

$$
\hat{y} = \text{softmax}(x W + b), \qquad W \in \mathbb{R}^{V \times V}
$$

Multiplying a one-hot vector with $W$ selects one row of $W$, so the model has one row of free parameters per word, just like the bigram table. Training the weights by gradient descent with the cross-entropy loss gives (up to the effect of the optimization) the same probabilities as counting.

What is gained is flexibility: we can put a **hidden layer** with a small number $d \ll V$ of neurons between the input and the output,

$$
\hat{y} = \text{softmax}\big( \varphi(x W_h + b_h) W_o + b_o \big)
$$

Now each word is first mapped to a short dense vector (the row of $W_h$ that belongs to the word, i.e. a *word embedding*), and the next-word probabilities are computed from this vector. Words that are followed by similar words end up with similar vectors, so statistical strength is shared across words and we need far fewer parameters ($2Vd$ instead of $V^2$). This is the idea behind the neural language models and the word embeddings that we study in this course.

To get there we need the building blocks of neural networks. The following chapters develop them one by one, from the simplest model to the bigram model as a neural network:

1. [Simple linear regression as a single neuron](03-03-ANN-Simple-Linear-Regression.ipynb): weights, bias, loss function and gradient descent.
2. [Multiple regression and hidden layers](03-04-ANN-Multiple-Regression.ipynb): matrix notation, activation functions and the gradients of a network with a hidden layer.
3. [Logistic regression with a hidden layer](03-05-ANN-Logistic-Regression.ipynb): entropy, the cross-entropy loss and classification with two classes.
4. [Multi-class classification](03-06-ANN-Multiclass-Classification.ipynb): the softmax function, its gradients, and finally the bigram model as a softmax network.

## A short history

The ideas in this chapter have a long history, which is worth knowing because the same questions keep returning in new forms.

- **Markov chains (1913).** Andrey Markov counted the vowels and consonants in the first 20,000 letters of Pushkin's *Eugene Onegin* and showed that whether a letter is a vowel or a consonant depends on the preceding letter [@MARKOV2006ExampleStatisticalInvestigation]. Such dependence of each element on the previous one is the first-order Markov assumption that we made above for words.
- **Information theory and n-grams (1948).** Claude Shannon introduced the entropy of a source and demonstrated how sequences of letters and words generated from n-gram statistics look more and more like English as $n$ grows [@SHANNON1948MathematicalTheoryCommunication]. The entropy and cross-entropy are the subject of the [logistic regression chapter](03-05-ANN-Logistic-Regression.ipynb).
- **Statistical language models (1980s to 2000s).** Word n-gram models with smoothing became the workhorse of speech recognition and machine translation; extensions such as class-based n-grams [@BROWN1992ClassBasedNgramModels] tackled the sparsity problem by grouping similar words. See @JURAFSKY2025SpeechLanguageProcessing for a textbook treatment.
- **Artificial neurons and learning (1943 to 1986).** The first mathematical model of a neuron [@MCCULLOCH1943LogicalCalculusIdeas], the perceptron with its learning rule [@ROSENBLATT1958PerceptronProbabilisticModel] and, importantly, the back-propagation algorithm for training networks with hidden layers [@RUMELHART1986LearningRepresentationsBackpropagating] are the foundations of the networks in the following chapters.
- **Neural language models (2003).** Bengio and co-authors combined the two lines of work: a neural network with a hidden layer learns word vectors and the next-word probabilities jointly, which overcomes the sparsity problem of n-gram models [@BENGIO2003NeuralProbabilisticLanguage].
- **Word embeddings and transformers (2013 and later).** Word2vec made it cheap to learn good word vectors from huge corpora [@MIKOLOV2013EfficientEstimationWord], and the transformer architecture [@VASWANI2017AttentionAllYou] replaced the fixed-size context by attention over the whole text.

## Implementation

We estimate the bigram probabilities from the complete text of *Alice's Adventures in Wonderland* (distributed with NLTK) by counting.

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg

nltk.download('gutenberg', quiet=True)

alice_text = gutenberg.raw('carroll-alice.txt')

# A very simple tokenizer: lower-case words (letters and apostrophes only)
tokens = re.findall(r"[a-z']+", alice_text.lower())

print(f"Number of tokens: {len(tokens)}")
print(f"Vocabulary size: {len(set(tokens))}")
print("First tokens:", tokens[:12])

Number of tokens: 27802
Vocabulary size: 2856
First tokens: ["alice's", 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'chapter', 'i', 'down', 'the', 'rabbit']


### The tiny corpus

First we reproduce the table from above: the corpus consists of a single sentence, so every row has at most one non-zero entry.

In [2]:
toy_sentence = "alice was beginning to get very tired".split()

toy_vocab = sorted(set(toy_sentence))
toy_index = {word: i for i, word in enumerate(toy_vocab)}

# Count how often each word (row) is followed by each other word (column)
toy_counts = np.zeros((len(toy_vocab), len(toy_vocab)))

for first_word, second_word in zip(toy_sentence[:-1], toy_sentence[1:]):
    toy_counts[toy_index[first_word], toy_index[second_word]] += 1

# Divide each row by its total (rows without any bigram stay zero)
row_totals = toy_counts.sum(axis=1, keepdims=True)
toy_probs = np.divide(toy_counts, row_totals, out=np.zeros_like(toy_counts), where=row_totals > 0)

pd.DataFrame(toy_probs, index=toy_vocab, columns=toy_vocab)

,alice,beginning,get,tired,to,very,was
alice,0.0,0.0,0.0,0.0,0.0,0.0,1.0
beginning,0.0,0.0,0.0,0.0,1.0,0.0,0.0
get,0.0,0.0,0.0,0.0,0.0,1.0,0.0
tired,0.0,0.0,0.0,0.0,0.0,0.0,0.0
to,0.0,0.0,1.0,0.0,0.0,0.0,0.0
very,0.0,0.0,0.0,1.0,0.0,0.0,0.0
was,0.0,1.0,0.0,0.0,0.0,0.0,0.0


### The whole book

Now we count all bigrams in the book. `first_counts[w]` is the number of times that `w` is followed by another word, which is the denominator $C(w_{t-1})$ of the estimator.

In [3]:
bigram_counts = Counter(zip(tokens[:-1], tokens[1:]))
first_counts = Counter(tokens[:-1])
unigram_counts = Counter(tokens)

def next_word_probs(word, top=5):
    """The `top` most likely next words and their estimated probabilities"""
    candidates = [(second, count / first_counts[word])
                  for (first, second), count in bigram_counts.items() if first == word]
    return sorted(candidates, key=lambda pair: -pair[1])[:top]

for word in ["alice", "the", "said"]:
    print(f"After '{word}':", [(w, round(p, 3)) for w, p in next_word_probs(word)])

After 'alice': [('was', 0.044), ('and', 0.031), ('thought', 0.031), ('had', 0.029), ('said', 0.029)]
After 'the': [('queen', 0.038), ('king', 0.037), ('mock', 0.033), ('gryphon', 0.033), ('hatter', 0.031)]
After 'said': [('the', 0.45), ('alice', 0.249), ('to', 0.084), ('in', 0.017), ('this', 0.013)]


### Scoring sentences

The log-probability of a sentence is the log of the probability of the first word plus the sum of the log bigram probabilities. A sentence with an unseen bigram gets probability zero (log-probability $-\infty$).

In [4]:
def sentence_log_prob(sentence):
    words = sentence.lower().split()
    log_prob = np.log(unigram_counts[words[0]] / len(tokens))

    for first, second in zip(words[:-1], words[1:]):
        if bigram_counts[(first, second)] == 0:
            return -np.inf     # unseen bigram: probability zero
        log_prob += np.log(bigram_counts[(first, second)] / first_counts[first])

    return log_prob

print("Natural order: ", sentence_log_prob("alice was beginning to get very tired"))
print("Reversed order:", sentence_log_prob("tired very get to beginning was alice"))

Natural order:  -23.67276080732909
Reversed order: -inf


### Generating text

We generate text by sampling the next word from the row of the (implicit) bigram matrix that belongs to the current word. The random number generator is seeded so that the output is reproducible.

In [5]:
rng = np.random.default_rng(1)

def sample_sentence(start_word, length=12):
    words = [start_word]
    for _ in range(length - 1):
        candidates = [(second, count) for (first, second), count in bigram_counts.items()
                      if first == words[-1]]
        if not candidates:
            break
        next_words, counts = zip(*candidates)
        words.append(rng.choice(next_words, p=np.array(counts) / sum(counts)))
    return " ".join(words)

for _ in range(3):
    print(sample_sentence("alice"))

alice come before she comes at all as it but it out
alice heard it put on shrinking rapidly she was the king 'and
alice said but come or two the door began ordering people began


### Sparsity

Finally, let's see how much of the $V \times V$ table we actually observe.

In [6]:
vocab_size = len(set(tokens))

print(f"Possible bigrams (V^2): {vocab_size ** 2:,}")
print(f"Observed bigrams: {len(bigram_counts):,}")
print(f"Share of the table that is observed: {len(bigram_counts) / vocab_size ** 2:.4%}")

Possible bigrams (V^2): 8,156,736
Observed bigrams: 15,071
Share of the table that is observed: 0.1848%
